In [32]:
import requests
import jwt
from comotion.dash import DashConfig, Query, Load
from comotion.auth import Auth

In [33]:
org_name = 'momentum'
entity_type = Auth.APPLICATION
application_client_id = "EXTERNAL_MOMENTUM_TEST_CREDENTIALS"
application_client_secret = "3C2wjEgPRWZyrSEgLnkACtr3vOo7oTap"
auth = Auth(orgname=org_name, entity_type=entity_type, application_client_id=application_client_id, application_client_secret=application_client_secret)
config = DashConfig(auth = auth)

# Get the access token and then decode it
access_token = auth.get_access_token()
access_token_decoded = jwt.decode(access_token, options={"verify_signature": False})
print(access_token_decoded)

{'exp': 1768213722, 'iat': 1768213422, 'jti': 'e05623e7-fb34-4a12-a901-5646436d7a40', 'iss': 'https://auth.comotion.us/auth/realms/momentum', 'aud': ['dash_api', 'momentum_client', 'dash_api_preprod'], 'sub': 'fe823a86-bb6c-430e-8f05-2b6ba46cc1a5', 'typ': 'Bearer', 'azp': 'EXTERNAL_MOMENTUM_TEST_CREDENTIALS', 'acr': '1', 'resource_access': {'dash_api': {'roles': ['AllServiceClientAnalyst', 'query_runner']}, 'momentum_client': {'roles': ['APIUser']}, 'dash_api_preprod': {'roles': ['AllServiceClientAnalyst', 'load_runner', 'query_runner']}}, 'scope': 'service_client preprod:load:write profile preprod:query:read email preprod:load:read query:write preprod:query:write query:read', 'email_verified': False, 'clientId': 'EXTERNAL_MOMENTUM_TEST_CREDENTIALS', 'clientHost': '197.221.188.2', 'preferred_username': 'service-account-external_momentum_test_credentials', 'service_client_id': ['0'], 'clientAddress': '197.221.188.2'}


In [27]:
create_table_sql = """
create table "momentum_sandbox".api_test as
with numbers as (
    select n
    from unnest(sequence(0, 999)) as t(n)
)
select
    n as id,
    case when n % 2 = 0 then 'Alice' else 'Bob' end as name,
    date_add('day', n, DATE '2000-01-01') as some_date,
    date_add('hour', n, date_parse('2023-01-01', '%Y-%m-%d')) as some_timestamp,
    round(n * 1.01, 2) as some_float,
    (n % 2 = 0) as is_active,
    cast(n % 100 as smallint) as small_val,
    cast((n % 1000) as bigint) as a_big_int,
    array['foo', cast(n as varchar)] as some_strings,
    map(array['k'], array[n*42]) as some_obj,
    nullif(n % 13, 0) as nullable_col,
    case
        when n % 10 < 3 then 'groupA'
        when n % 10 < 6 then 'groupB'
        else 'groupC'
    end as label
from numbers
"""

query = Query(config= config, query_text = create_table_sql)


result = query.wait_to_complete()
print(result.status)
print(result.status.state)


state='FAILED' state_change_reason="TABLE_ALREADY_EXISTS: line 1:1: Destination table 'awsdatacatalog.momentum_sandbox.api_test' already exists. " submission_date_time='01-12-2026 10:08:17' completion_date_time='01-12-2026 10:08:17'
FAILED


In [34]:
query = Query(config= config, query_text = """select * from momentum_sandbox.api_test""")
result = query.wait_to_complete()
print(result.status)
print(result.status.state)

state='SUCCEEDED' state_change_reason=None submission_date_time='01-12-2026 10:23:48' completion_date_time='01-12-2026 10:23:49'
SUCCEEDED


In [16]:
create_table_sql = """drop table if exists momentum_sandbox.api_test"""

query = Query(config= config, query_text = create_table_sql)
# query = Query(config= config, query_text = """select 1""")


result = query.wait_to_complete()
print(result.status)
print(result.status.state)

state='SUCCEEDED' state_change_reason=None submission_date_time='01-12-2026 08:05:35' completion_date_time='01-12-2026 08:05:35'
SUCCEEDED


In [12]:
from comodash_api_client_lowlevel import ApiClient, QueriesApi

# Reuse the same config you used to build `query`
api_client = ApiClient(query.config)
qa = QueriesApi(api_client)

# Fetch the first page of results
qr = qa.get_query_results(query_id=query.query_id)

# Easiest: just print each row as a list of values
for row in qr.result_set.rows:
    print([cell.var_char_value for cell in row.data])

ServiceException: (500)
Reason: Internal Server Error
HTTP response headers: HTTPHeaderDict({'Date': 'Fri, 09 Jan 2026 16:39:42 GMT', 'Content-Type': 'text/plain; charset=utf-8', 'Content-Length': '96', 'Connection': 'keep-alive', 'Apigw-Requestid': 'W7RT8g1yoAMEYrA='})
HTTP response body: {"message": "An Internal Server Error occurred. Please contact support if this error persists."}
